# Local Models with Ollama

Run the entire catchfly pipeline without sending data to external APIs. Your documents never leave your machine.

**Ideal for:** Protected health information (PHI), air-gapped environments, cost sensitivity, development iteration.

**Estimated API cost:** $0.00

## Prerequisites

Install [Ollama](https://ollama.com) and pull a model:

```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama pull qwen3.5
```

For structured extraction, use models with good instruction following: `qwen3.5`, `llama3.1:8b`, `mistral`, `gemma2:9b`.

In [ ]:
# pip install catchfly
# No API key needed

## Run the pipeline

The only difference from cloud API usage: `model` and `base_url` parameters. Auto field selection works the same way — `StatisticalFieldSelector` decides which fields to normalize.

In [ ]:
from catchfly import Pipeline
from catchfly.demo import load_samples

docs = load_samples("case_reports")

pipeline = Pipeline.quick(
    model="qwen3.5",
    base_url="http://localhost:11434/v1",
)

# No normalize_fields needed — auto field selection works with local models too
results = pipeline.run(
    docs[:3],  # start small to test
    domain_hint="Rare disease clinical case reports",
)

print(f"Auto-normalized fields: {list(results.normalizations.keys())}")
for record in results.records:
    print(f"  {record.diagnosis}")

Schema discovery, extraction, field selection, and normalization all ran locally through Ollama's OpenAI-compatible endpoint.

## Modular usage with local models

### Discovery

In [ ]:
from catchfly.discovery import SinglePassDiscovery

discovery = SinglePassDiscovery(
    model="qwen3.5",
    base_url="http://localhost:11434/v1",
)
schema = discovery.discover(docs[:3], domain_hint="Medical case reports")

print("Discovered fields:")
for name in schema.json_schema["properties"]:
    print(f"  - {name}")

### Extraction

In [ ]:
from catchfly.extraction import LLMDirectExtraction

extractor = LLMDirectExtraction(
    model="qwen3.5",
    base_url="http://localhost:11434/v1",
    max_retries=2,
    on_error="skip",  # local models may need more tolerance
)
result = extractor.extract(schema.model, docs[:3])
print(f"Extracted {len(result.records)} records")

### Normalization

In [ ]:
from catchfly.normalization import LLMCanonicalization

normalizer = LLMCanonicalization(
    model="qwen3.5",
    base_url="http://localhost:11434/v1",
)
norm = normalizer.normalize(
    ["seizures", "epileptic fits", "convulsions", "hepatomegaly", "liver enlargement"],
    context_field="phenotypes",
)
for raw, canonical in norm.mapping.items():
    print(f"  {raw} -> {canonical}")

## Quality comparison

Local models trade quality for privacy:

| Aspect | Local (8B model) | API (GPT-5.4-mini) |
|--------|-----------------|---------------------|
| Schema discovery | Good for common domains | Excellent across domains |
| Extraction accuracy | 70-85% on medical text | 90-95% on medical text |
| Normalization quality | Groups obvious variants | Groups semantic synonyms |
| Speed | 5-15 tokens/sec (CPU) | 50-100 tokens/sec |
| Cost | $0.00 | ~$0.01 per document |

### Hybrid approach

Use local models for development and iteration, then switch to API models for the final production run. The code change is just two parameters:

```python
# Development (local)
pipeline = Pipeline.quick(model="qwen3.5", base_url="http://localhost:11434/v1")

# Production (API)
pipeline = Pipeline.quick(model="gpt-5.4-mini")
```

## Other local providers

Catchfly works with any OpenAI-compatible endpoint:

```python
# vLLM
pipeline = Pipeline.quick(model="meta-llama/Llama-3.1-8B-Instruct", base_url="http://localhost:8000/v1")

# LM Studio
pipeline = Pipeline.quick(model="local-model", base_url="http://localhost:1234/v1")

# Text Generation Inference (TGI)
pipeline = Pipeline.quick(model="tgi", base_url="http://localhost:8080/v1")
```